# Optimizer Comparison: Momentum vs AdaGrad vs RMSProp vs Adam

**Setup**
- Task: binary classification (10,000 samples, 20 features) on a synthetic dataset
- Model: small MLP with **Batch Normalization** layers
- Training: **mini-batch** gradient descent
- Loss: **Binary Cross-Entropy** (`BCEWithLogitsLoss`)
- Every optimizer starts from the **same initial weights** and sees the **same batches**, so only the update rule differs
- Results are averaged over several seeds to reduce noise

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 1. Dataset

In [ ]:
X, y = make_classification(
    n_samples=10_000, n_features=20, n_informative=12, n_redundant=4,
    n_clusters_per_class=2, flip_y=0.03, class_sep=1.0, random_state=0
)

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0, stratify=y_tmp)

scaler = StandardScaler().fit(X_train)
X_train, X_val, X_test = [scaler.transform(a) for a in (X_train, X_val, X_test)]

def to_tensor(X, y):
    return (torch.tensor(X, dtype=torch.float32, device=device),
            torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1))

Xtr, ytr = to_tensor(X_train, y_train)
Xva, yva = to_tensor(X_val, y_val)
Xte, yte = to_tensor(X_test, y_test)

print(f"Train: {len(Xtr)} | Val: {len(Xva)} | Test: {len(Xte)}")
print(f"Class balance (train): {y_train.mean():.2f}")

## 2. Model, optimizers and training loop

In [ ]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(20, 64), nn.BatchNorm1d(64), nn.ReLU(),
        nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(),
        nn.Linear(32, 1)          # raw logit -> BCEWithLogitsLoss applies the sigmoid
    ).to(device)

# Learning rates are the usual defaults for each optimizer (tune them for a fairer comparison)
OPTIMIZERS = {
    "Momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "AdaGrad":  lambda p: torch.optim.Adagrad(p, lr=0.01),
    "RMSProp":  lambda p: torch.optim.RMSprop(p, lr=0.001, alpha=0.9),
    "Adam":     lambda p: torch.optim.Adam(p, lr=0.001, betas=(0.9, 0.999)),
}

BATCH_SIZE = 64
EPOCHS = 30
SEEDS = [0, 1, 2]

criterion = nn.BCEWithLogitsLoss()

@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    logits = model(X)
    loss = criterion(logits, y).item()
    acc = ((logits > 0).float() == y).float().mean().item()
    return loss, acc

def train(opt_name, seed):
    model = make_model(seed)                       # identical init for every optimizer (same seed)
    opt = OPTIMIZERS[opt_name](model.parameters())
    g = torch.Generator().manual_seed(seed)        # identical batch order for every optimizer
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH_SIZE, shuffle=True, generator=g)

    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(EPOCHS):
        model.train()
        running, n = 0.0, 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            running += loss.item() * len(xb)
            n += len(xb)
        vl, va = evaluate(model, Xva, yva)
        hist["train_loss"].append(running / n)
        hist["val_loss"].append(vl)
        hist["val_acc"].append(va)
    test_loss, test_acc = evaluate(model, Xte, yte)
    return hist, test_loss, test_acc

## 3. Run the experiment

In [ ]:
results = {name: {"train_loss": [], "val_loss": [], "val_acc": [], "test_loss": [], "test_acc": []}
           for name in OPTIMIZERS}

for name in OPTIMIZERS:
    for seed in SEEDS:
        hist, tl, ta = train(name, seed)
        for k in hist:
            results[name][k].append(hist[k])
        results[name]["test_loss"].append(tl)
        results[name]["test_acc"].append(ta)
    print(f"{name:9s} done | test acc = {np.mean(results[name]['test_acc']):.4f}")

## 4. Learning curves (mean ± std over seeds)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
panels = [("train_loss", "Train BCE loss"), ("val_loss", "Validation BCE loss"), ("val_acc", "Validation accuracy")]

for ax, (key, title) in zip(axes, panels):
    for name in OPTIMIZERS:
        arr = np.array(results[name][key])          # (seeds, epochs)
        m, s = arr.mean(0), arr.std(0)
        x = np.arange(1, len(m) + 1)
        ax.plot(x, m, label=name)
        ax.fill_between(x, m - s, m + s, alpha=0.2)
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout()
plt.show()

## 5. Final results

In [ ]:
rows = []
for name in OPTIMIZERS:
    r = results[name]
    best_epoch = np.array(r["val_loss"]).mean(0).argmin() + 1
    rows.append({
        "Optimizer": name,
        "Final train loss": np.mean([h[-1] for h in r["train_loss"]]),
        "Final val loss":   np.mean([h[-1] for h in r["val_loss"]]),
        "Best val epoch":   best_epoch,
        "Test loss":        np.mean(r["test_loss"]),
        "Test acc (mean)":  np.mean(r["test_acc"]),
        "Test acc (std)":   np.std(r["test_acc"]),
    })

df = pd.DataFrame(rows).set_index("Optimizer").round(4)
df

## 6. Things to try next
- **Tune learning rates**: each optimizer has a different sweet spot, and a fair comparison needs a small LR sweep for each (e.g. `[1e-3, 3e-3, 1e-2, 3e-2]`).
- **Change the batch size** (16 / 64 / 256) and see which optimizers are most sensitive.
- **Remove BatchNorm** and see how the ranking changes.
- Expect AdaGrad to slow down late in training, since its accumulated squared gradients only grow and the effective learning rate shrinks.